# Basic NoteBook example for Radiopyo module usage

## Module imports

In [ ]:
# External modules (to "pip install")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# -> RadioBio module
import radiopyo as rp

## Simple Base Case

### Load configuration from module's built-in data

In [ ]:
# Model "Labarbe like" i.e. with some  biology stuffs
uc = rp.UnitCell.from_toml(rp.LABARBE_CONFIG_FILE)

### Run the simulation

In [ ]:
res = uc.run(t_span=[1e-7, 10]) #Run simulation from 1e-7 s to 10 s.

### Export time evolution concentrations data to a pandas DataFrame

In [ ]:
df = res.to_pandas() 

In [ ]:
# Time integrate species concentrations.
s = res.integrate_species() # pandas.Series

### ReConstruct the reaction rates at each time steps

In [ ]:
rdf = res.reconstruct_reactions()

### Playing with the Chemical Environment to get more data

In [ ]:
## Get reactions labels (as in the DataFrame)
for r in uc.env.reactions.iter_all():
    print(f"{r.as_label()}) {r}")

## Get a Specific reactions from simulation Env
r = uc.env.reactions_involving("H2O2") # -> List of Tuple[Reaction, float], the float is reaction constant 
# or be more specific
r = uc.env.reactions_involving_product("H2O2")

## Get a specific species
sp = uc.env.species.get("H2O2")
print(sp.label, " --> ", sp.index)

## More advanced beam definition and remove the biology (-> Pure Water)

In [ ]:
uc = rp.UnitCell.from_toml(rp.PURE_WATER_CONFIG_FILE)

### Define a new Pulsed Beam

In [ ]:
peak_dose_rate  = 1000 # Gy/s (dose rate during ON time)
max_dose = 10 # Gy
period = 10e-3 # seconds
on_time = 1e-3 # seconds (=> Duty cycle of 10%)
beam = rp.PulsedBeam.from_dose_rate_peak(peak_dose_rate, period, on_time, max_dose)

### Set new beam 

In [ ]:
uc.set_beam(beam, LET=0.2)

### Set O2 Initial Concentration

In [ ]:
sp = uc.env.species.get("O2")
sp.set_initial_cc(52e-6) # 4% pO2
# Then update sim env
uc.update_env()
# Ready to simulate

### Run the multi-pulsed simulation in several simulation chunks

In [ ]:
# Better to adapt simulation settings during ON and OFF time.
# /!\ For atol, the base unit for the ODE solver is [µmol/l]
sim = uc.prepare_chunked_run(t_span=[1e-7, 10],
    atol=1e-4,              # Absolute tolerance of the ODE Solver
    rtol=1e-6,              # Relative tolerance of the ODE Solver
    max_step_size_on=1e-5,  # Seconds (on = during beam ON)
    max_step_size_off=0.01, # Seconds (off = during beam OFF)
    max_step_size_final=1,  # Seconds (final = after complete dose delivery)
)
res = sim.run()

### Time to make some plots

In [ ]:
fig, ax = plt.subplots()

ax.plot(res.time, res["O2"], marker="")

ax.set_xlabel("Time [s]")
ax.set_ylabel("$O_2$ [µM]")
ax.set_xscale("log")
ax.grid()

ax2 = ax.twinx()
ax2.plot(res.time, -np.log10(res["H3O_plus"]*1e-6), marker="", color="#e41a1c")
ax2.set_ylabel("pH")
